In [1]:
# Import module constrained_likelihood_surrogates
#
import sys
sys.path.append("../src/")
from constrained_likelihood_surrogates import *

In [ ]:
# Checking COMPUTATIONAL TIME and RATE OF REJECTION from constrained and typical surrogates:
#
# Varying num_trans for fixed N
# 
# Use several types of statistics as test statistics
# 
# 
# Model-free statistics:
free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth]
# free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth, log_mean, log_var, log_skew, log_kurt, jb_stat, log_jb_stat, dap_stat, log_dap_stat]
# free_stat_list = free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth, geom_mean, harm_mean, rang, log_rang, iq_rang, log_iq_rang]
num_free_stats = len(free_stat_list)
# Model-dependent statistics (dependent on one model - this characterisation is an excuse to leave out the likelihood ratio):
# depe_stat_list = [ks_stat, ad_stat, cvm_stat]
depe_stat_list = [ks_stat, kuiper_stat, ad_stat, cvm_stat, zk_stat, za_stat, zc_stat]
num_depe_stats = len(depe_stat_list)

test_type = 'power'
lower_cutoff = 1
for N in [2**6, 2**9]:
    num_trans_list = [N, 2*N, 4*N, 8*N, 16*N, 32*N, 64*N]
    for num_trans in num_trans_list:
    
        num_surr = 19
        num_tests = 10**3
    
        a = lower_cutoff
        b = 9
    
        print('N = ' + str(N) + ', lower cut-off = ' + str(lower_cutoff))
    
        print(str(num_tests) + ' tests, each with ' + str(num_surr) + ' constrained, typical surrogates, each using ' + str(num_trans) + ' transitions')
    
        # process_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform', 'powerlaw_t', 'lognorm_t', 'expon_t', 'truncnorm_t', 'uniform_t']
        process_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
        # process_list = ['powerlaw', 'lognorm', 'uniform']
        # process_list = ['expon', 'truncnorm', 'uniform']
        # process_list = ['expon', 'truncnorm', 'uniform']
        # process_list = ['powerlaw']
        num_processes = len(process_list)
    
        model_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
        model_list = process_list
        # model_list = ['truncnorm']
        num_models = len(model_list)
    
        num_methods = 2#Based on constrained, typical
    
        free_quantile_list_list_list = np.full((num_methods, num_processes, num_models, num_tests, num_free_stats), np.nan)#Array of nans: num_methods x num_processes x num_models x num_tests x num_free_stats
        depe_quantile_list_list_list = np.full((num_methods, num_processes, num_models, num_tests, num_depe_stats), np.nan)#Array of nans: num_methods x num_processes x num_models x num_tests x num_depe_stats
        surr_time_list_list_list = np.full((num_methods, num_processes, num_models, num_tests), np.nan)#Average time per surrogate: num_methods x num_processes x num_models x num_tests
        param_fit_time_list_list_list = np.full((num_processes, num_models, num_tests), np.nan)#Time to calculate maximum likelihood parameters: num_processes x num_models x num_tests
        save_str_0 = 'time_quant_free-depe' + '_' + test_type + '_xmin-' + str(lower_cutoff) + '_N-' + str(N) + '_ntra-' + str(num_trans) + '_nsur-' + str(num_surr) + '_ntes-' + str(num_tests) + '_ntyp-' + str(num_methods) + '_npro-' + str(num_processes) + '_nmod-' + str(num_models) + '_nfs-' + str(num_free_stats) + '_nds-' + str(num_depe_stats)
    
        process_str = ''
        process_dict = {}
        process_time_list = []
    
        for i_process in range(num_processes):
            start_time_process = timer()
            process = process_list[i_process]
    
            if (process == 'powerlaw'):#Generate power law sequence
                alpha = 1.5
                param = [alpha]
                param_str = '[alpha]'
    
            elif (process == 'lognorm'):#Generate truncated log normal sequence
                mu = -1
                sigma = 1
                param = [mu, sigma]
                param_str = '[mu/alpha, sigma]'
    
            elif (process == 'expon'):#Generate exponential sequence
                lam = 1
                param = [lam]
                param_str = '[lambda]'
    
            elif (process == 'truncnorm'):#Generate truncated normal sequence
                mu = -1
                sigma = 1
                param = [mu, sigma]
                param_str = '[mu/lambda, sigma]'
    
            elif (process == 'uniform'):#Generate uniform sequence
                param = [b]
                param_str = '[b]'
    
            if (process == 'powerlaw_t'):#Generate tightly bounded power law sequence
                alpha = 1.5
                param = [alpha, a, b]
                param_str = '[alpha, a, b]'
    
            elif (process == 'lognorm_t'):#Generate tightly bounded truncated log normal sequence
                mu = -1
                sigma = 1
                param = [mu, sigma, a, b]
                param_str = '[mu/alpha, sigma, a, b]'
    
            elif (process == 'expon_t'):#Generate tightly bounded exponential sequence
                lam = 1
                param = [lam, a, b]
                param_str = '[lambda, a, b]'
    
            elif (process == 'truncnorm_t'):#Generate tightly bounded truncated normal sequence
                mu = -1
                sigma = 1
                param = [mu, sigma, a, b]
                param_str = '[mu/lambda, sigma, a, b]'
    
            elif (process == 'uniform_t'):#Generate tightly bounded uniform sequence
                param = [b, a, b]
                param_str = '[b, a, b]'
    
            print('Process is ' + process)
            print('True ' + param_str + ' = ' + str(param))
    
            process_dict[process] = [param, param_str]
    
            param_str_2 = str(param)
            param_str_2 = param_str_2.replace(', ', '-')
            param_str_2 = param_str_2[1:-1]
            if (i_process < num_processes):
                process_str = process_str + '_'
            process_str = process_str + process[0] + '-' + param_str_2
    
            free_quantile_power_list_list = np.full((num_methods, num_models, num_tests, num_free_stats), np.nan)#Array of nans
            depe_quantile_power_list_list = np.full((num_methods, num_models, num_tests, num_depe_stats), np.nan)#Array of nans
    
            for i_test in range(num_tests):
                #Generate time series and calculate ks distance:
                val_seq = gen_data(process, N=N, lower_cutoff=lower_cutoff, param=param)
                lower_cutoff_hat = lower_cutoff
                free_stat_val_list = [stat(val_seq) for stat in free_stat_list]
                for i_model in range(num_models):
                    model = model_list[i_model]
    
                    start_time_param_fit = timer()
                    param_hat_m, lp_seq_m = fit_model(val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)#m for model
                    end_time_param_fit = timer()
                    param_fit_time_list_list_list[i_process, i_model, i_test] = end_time_param_fit - start_time_param_fit
    
                    val_seq_sorted = sorted(val_seq)
                    emp_cdf_with_rep = rankdata(val_seq_sorted, method='max')/len(val_seq_sorted)
                    cdf_fun = return_cdf_func(param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                    exp_cdf_with_rep = cdf_fun(val_seq_sorted)
                    depe_stat_val_list = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
    
                    c_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Constrained
                    t_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Typical
    
                    c_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Constrained
                    t_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Typical
    
                    surr_m_val_seq = val_seq
                    surr_m_val_seq_list = []
                    start_time_c_surr = timer()
                    for i_surr in range(num_surr):
                        method = 'constrained'
                        surr_m_val_seq = gen_surrogate(val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)#Each surrogate generated independently from the original sequence
                        # surr_m_val_seq = gen_surrogate(surr_m_val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)#Each surrogate generated from the previous surrogate
                        random.shuffle(surr_m_val_seq)
                        surr_m_val_seq_list = surr_m_val_seq_list + [surr_m_val_seq]
                    end_time_c_surr = timer()
                    surr_time_list_list_list[0, i_process, i_model, i_test] = (end_time_c_surr - start_time_c_surr)/num_surr
    
                    for i_surr in range(num_surr):
                        method = 'constrained'
                        surr_m_val_seq = surr_m_val_seq_list[i_surr]
                        surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                        surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                        c_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                        surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                        emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                        cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                        exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                        surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                        c_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m
    
                    t_surr_m_val_seq_list = []
                    start_time_t_surr = timer()
                    for i_surr in range(num_surr):
                        method = 'typical'
                        surr_m_val_seq = gen_surrogate(val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)
                        t_surr_m_val_seq_list = t_surr_m_val_seq_list + [surr_m_val_seq]
                    end_time_t_surr = timer()
                    surr_time_list_list_list[1, i_process, i_model, i_test] = (end_time_t_surr - start_time_t_surr)/num_surr
    
                    for i_surr in range(num_surr):
                        method = 'typical'
                        surr_m_val_seq = t_surr_m_val_seq_list[i_surr]
                        surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                        surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                        t_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                        surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                        emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                        cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                        exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                        surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                        t_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m
                        if (surr_m_depe_stat_val_list_m[0] > 1):
                            raise Exception('Impossibly high value of the KS distance')
    
                    for i_free_stat in range(num_free_stats):
                        #Calculate values of discriminating statistic (model-free statistics) for observed sequence and surrogate sequences:
                        obs_stat = free_stat_val_list[i_free_stat]
                        abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                        if (abs_obs_stat == np.inf):
                            abs_obs_stat = 0
                        obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                        if np.isnan(obs_stat):
                            print(f'There may be a problem with model-free statistic no. {i_free_stat} (observed)')
                        for i_method in range(num_methods):
                            if (i_method == 0):#Constrained surrogates
                                stat_val_surr_m_list = c_surr_m_free_list_list[:, i_free_stat]
                            elif (i_method == 1):#Typical surrogates
                                stat_val_surr_m_list = t_surr_m_free_list_list[:, i_free_stat]
                            stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                            if np.any(np.isnan(stat_val_surr_m_list)):
                                print(f'There may be a problem with model-free statistic no. {i_free_stat} (surrogate)')
                            #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                            rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                            rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                            r = random.randint(rankMin, rankMax)
                            q = (r - 0.5)/(num_surr + 1)
                            free_quantile_power_list_list[i_method, i_model, i_test, i_free_stat] = q
                            free_quantile_list_list_list[i_method, i_process, i_model, i_test, i_free_stat] = q
                            if (np.isnan(q)):
                                raise Exception('Calculated quantile q is not a number.')
                                
                    for i_depe_stat in range(num_depe_stats):
                        #Calculate values of discriminating statistic (model-dependent statistics) for observed sequence and surrogate sequences:
                        obs_stat = depe_stat_val_list[i_depe_stat]
                        abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                        if (abs_obs_stat == np.inf):
                            abs_obs_stat = 0
                        obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                        if np.isnan(obs_stat):
                            print(f'There may be a problem with model-dependent statistic no. {i_depe_stat} (observed)')
                        for i_method in range(num_methods):
                            if (i_method == 0):#Constrained surrogates
                                stat_val_surr_m_list = c_surr_m_depe_list_list[:, i_depe_stat]
                            elif (i_method == 1):#Typical surrogates
                                stat_val_surr_m_list = t_surr_m_depe_list_list[:, i_depe_stat]
                            stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                            if np.any(np.isnan(stat_val_surr_m_list)):
                                print(f'There may be a problem with model-dependent statistic no. {i_depe_stat}  (surrogate)')
                            #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                            rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                            rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                            r = random.randint(rankMin, rankMax)
                            q = (r - 0.5)/(num_surr + 1)
                            depe_quantile_power_list_list[i_method, i_model, i_test, i_depe_stat] = q
                            depe_quantile_list_list_list[i_method, i_process, i_model, i_test, i_depe_stat] = q
                            if (np.isnan(q)):
                                raise Exception('Calculated quantile q is not a number.')
    
            end_time_process = timer()
            total_time_process = end_time_process - start_time_process
            process_time_list = process_time_list + [total_time_process]
            print('That took ' + str(total_time_process) + 'sec.') # Time in seconds, e.g. 5.38091952400282
            
        save_str = save_str_0 + process_str
    
        free_stat_name_list = [stat.__name__ for stat in free_stat_list]
        depe_stat_name_list = [stat.__name__ for stat in depe_stat_list]
    
        save_dict = {'free_quantile_list_list_list':free_quantile_list_list_list.tolist(),
                     'depe_quantile_list_list_list':depe_quantile_list_list_list.tolist(),
                     'surr_time_list_list_list':surr_time_list_list_list.tolist(),
                     'param_fit_time_list_list_list':param_fit_time_list_list_list.tolist(),
                     'test_type':test_type,
                     'lower_cutoff':lower_cutoff,
                     'N':N,
                     'num_trans':num_trans,
                     'num_surr':num_surr,
                     'num_tests':num_tests,
                     'num_methods':num_methods,
                     'process_list':process_list,
                     'process_dict':process_dict,
                     'process_str':process_str,
                     'model_list':model_list,
                     'save_str':save_str,
                     'process_time_list':process_time_list,
                     'free_stat_name_list':free_stat_name_list,
                     'depe_stat_name_list':depe_stat_name_list,
        }
    
        save('./results/hyp-test/benchmarking/' + save_str, save_dict)

In [2]:
# Checking COMPUTATIONAL TIME and RATE OF REJECTION of constrained and typical surrogates:
#
# Varying N
# 
# Use several types of statistics as test statistics
# 
# 
# Model-free statistics:
free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth]
# free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth, log_mean, log_var, log_skew, log_kurt, jb_stat, log_jb_stat, dap_stat, log_dap_stat]
# free_stat_list = free_stat_list = [mean_val, second_mom, third_mom, fourth_mom, max_val, variance, coef_var, dispersion, skew, kurtosis, evi_mom, evi_smooth, geom_mean, harm_mean, rang, log_rang, iq_rang, log_iq_rang]
num_free_stats = len(free_stat_list)
# Model-dependent statistics (dependent on one model - this characterisation is an excuse to leave out the likelihood ratio):
# depe_stat_list = [ks_stat, ad_stat, cvm_stat]
depe_stat_list = [ks_stat, kuiper_stat, ad_stat, cvm_stat, zk_stat, za_stat, zc_stat]
num_depe_stats = len(depe_stat_list)

test_type = 'power'
lower_cutoff = 1
# N_list = [2**n for n in range(2, 4)]
# N_list = [2**n for n in range(2, 11)]
N_list = [2**n for n in range(10, 11)]
for N in N_list:
    num_trans = N*(int(np.ceil(np.log2(N*1024))))

    num_surr = 19
    num_tests = 10**3

    a = lower_cutoff
    b = 9

    print('N = ' + str(N) + ', lower cut-off = ' + str(lower_cutoff))

    print(str(num_tests) + ' tests, each with ' + str(num_surr) + ' constrained and typical surrogates; constrained surrogates use ' + str(num_trans) + ' transitions')

    # process_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform', 'powerlaw_t', 'lognorm_t', 'expon_t', 'truncnorm_t', 'uniform_t']
    process_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
    # process_list = ['powerlaw', 'lognorm', 'uniform']
    # process_list = ['expon', 'truncnorm', 'uniform']
    # process_list = ['expon', 'truncnorm', 'uniform']
    # process_list = ['powerlaw']
    num_processes = len(process_list)

    model_list = ['powerlaw', 'lognorm', 'expon', 'truncnorm', 'uniform']
    model_list = process_list
    # model_list = ['truncnorm']
    num_models = len(model_list)

    num_methods = 2#Based on constrained, typical

    free_quantile_list_list_list = np.full((num_methods, num_processes, num_models, num_tests, num_free_stats), np.nan)#Array of nans: num_methods x num_processes x num_models x num_tests x num_free_stats
    depe_quantile_list_list_list = np.full((num_methods, num_processes, num_models, num_tests, num_depe_stats), np.nan)#Array of nans: num_methods x num_processes x num_models x num_tests x num_depe_stats
    surr_time_list_list_list = np.full((num_methods, num_processes, num_models, num_tests), np.nan)#Average time per surrogate: num_methods x num_processes x num_models x num_tests
    param_fit_time_list_list_list = np.full((num_processes, num_models, num_tests), np.nan)#Time to calculate maximum likelihood parameters: num_processes x num_models x num_tests
    save_str_0 = 'time_quant_free-depe' + '_' + test_type + '_xmin-' + str(lower_cutoff) + '_N-' + str(N) + '_ntra-' + str(num_trans) + '_nsur-' + str(num_surr) + '_ntes-' + str(num_tests) + '_ntyp-' + str(num_methods) + '_npro-' + str(num_processes) + '_nmod-' + str(num_models) + '_nfs-' + str(num_free_stats) + '_nds-' + str(num_depe_stats)

    process_str = ''
    process_dict = {}
    process_time_list = []

    for i_process in range(num_processes):
        start_time_process = timer()
        process = process_list[i_process]

        if (process == 'powerlaw'):#Generate power law sequence
            alpha = 1.5
            param = [alpha]
            param_str = '[alpha]'

        elif (process == 'lognorm'):#Generate truncated log normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma]
            param_str = '[mu/alpha, sigma]'

        elif (process == 'expon'):#Generate exponential sequence
            lam = 1
            param = [lam]
            param_str = '[lambda]'

        elif (process == 'truncnorm'):#Generate truncated normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma]
            param_str = '[mu/lambda, sigma]'

        elif (process == 'uniform'):#Generate uniform sequence
            param = [b]
            param_str = '[b]'

        if (process == 'powerlaw_t'):#Generate tightly bounded power law sequence
            alpha = 1.5
            param = [alpha, a, b]
            param_str = '[alpha, a, b]'

        elif (process == 'lognorm_t'):#Generate tightly bounded truncated log normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma, a, b]
            param_str = '[mu/alpha, sigma, a, b]'

        elif (process == 'expon_t'):#Generate tightly bounded exponential sequence
            lam = 1
            param = [lam, a, b]
            param_str = '[lambda, a, b]'

        elif (process == 'truncnorm_t'):#Generate tightly bounded truncated normal sequence
            mu = -1
            sigma = 1
            param = [mu, sigma, a, b]
            param_str = '[mu/lambda, sigma, a, b]'

        elif (process == 'uniform_t'):#Generate tightly bounded uniform sequence
            param = [b, a, b]
            param_str = '[b, a, b]'

        print('Process is ' + process)
        print('True ' + param_str + ' = ' + str(param))

        process_dict[process] = [param, param_str]

        param_str_2 = str(param)
        param_str_2 = param_str_2.replace(', ', '-')
        param_str_2 = param_str_2[1:-1]
        if (i_process < num_processes):
            process_str = process_str + '_'
        process_str = process_str + process[0] + '-' + param_str_2

        free_quantile_power_list_list = np.full((num_methods, num_models, num_tests, num_free_stats), np.nan)#Array of nans
        depe_quantile_power_list_list = np.full((num_methods, num_models, num_tests, num_depe_stats), np.nan)#Array of nans

        for i_test in range(num_tests):
            #Generate time series and calculate ks distance:
            val_seq = gen_data(process, N=N, lower_cutoff=lower_cutoff, param=param)
            lower_cutoff_hat = lower_cutoff
            free_stat_val_list = [stat(val_seq) for stat in free_stat_list]
            for i_model in range(num_models):
                model = model_list[i_model]

                start_time_param_fit = timer()
                param_hat_m, lp_seq_m = fit_model(val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)#m for model
                end_time_param_fit = timer()
                param_fit_time_list_list_list[i_process, i_model, i_test] = end_time_param_fit - start_time_param_fit

                val_seq_sorted = sorted(val_seq)
                emp_cdf_with_rep = rankdata(val_seq_sorted, method='max')/len(val_seq_sorted)
                cdf_fun = return_cdf_func(param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                exp_cdf_with_rep = cdf_fun(val_seq_sorted)
                depe_stat_val_list = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]

                c_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Constrained
                t_surr_m_free_list_list = np.zeros(shape=(num_surr, num_free_stats))#Typical

                c_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Constrained
                t_surr_m_depe_list_list = np.zeros(shape=(num_surr, num_depe_stats))#Typical

                surr_m_val_seq = val_seq
                surr_m_val_seq_list = []
                start_time_c_surr = timer()
                for i_surr in range(num_surr):
                    method = 'constrained'
                    surr_m_val_seq = gen_surrogate(val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)#Each surrogate generated independently from the original sequence
                    # surr_m_val_seq = gen_surrogate(surr_m_val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)#Each surrogate generated from the previous surrogate
                    random.shuffle(surr_m_val_seq)
                    surr_m_val_seq_list = surr_m_val_seq_list + [surr_m_val_seq]
                end_time_c_surr = timer()
                surr_time_list_list_list[0, i_process, i_model, i_test] = (end_time_c_surr - start_time_c_surr)/num_surr

                for i_surr in range(num_surr):
                    method = 'constrained'
                    surr_m_val_seq = surr_m_val_seq_list[i_surr]
                    surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                    surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                    c_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                    surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                    emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                    cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                    exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                    surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                    c_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m

                t_surr_m_val_seq_list = []
                start_time_t_surr = timer()
                for i_surr in range(num_surr):
                    method = 'typical'
                    surr_m_val_seq = gen_surrogate(val_seq, model=model, method=method, num_trans=num_trans, lower_cutoff_hat=lower_cutoff_hat, param_hat=param_hat_m)
                    t_surr_m_val_seq_list = t_surr_m_val_seq_list + [surr_m_val_seq]
                end_time_t_surr = timer()
                surr_time_list_list_list[1, i_process, i_model, i_test] = (end_time_t_surr - start_time_t_surr)/num_surr

                for i_surr in range(num_surr):
                    method = 'typical'
                    surr_m_val_seq = t_surr_m_val_seq_list[i_surr]
                    surr_m_param_hat_m, surr_m_lp_seq_m = fit_model(surr_m_val_seq, lower_cutoff_hat=lower_cutoff_hat, model=model)
                    surr_m_free_stat_list_m = [stat(surr_m_val_seq) for stat in free_stat_list]
                    t_surr_m_free_list_list[i_surr, :] = surr_m_free_stat_list_m
                    surr_m_val_seq_sorted = sorted(surr_m_val_seq)
                    emp_cdf_with_rep = rankdata(surr_m_val_seq_sorted, method='max')/len(surr_m_val_seq_sorted)
                    cdf_fun = return_cdf_func(surr_m_param_hat_m, model=model, lower_cutoff_hat=lower_cutoff_hat)
                    exp_cdf_with_rep = cdf_fun(surr_m_val_seq_sorted)
                    surr_m_depe_stat_val_list_m = [stat(exp_cdf_with_rep, emp_cdf_with_rep) for stat in depe_stat_list]
                    t_surr_m_depe_list_list[i_surr, :] = surr_m_depe_stat_val_list_m
                    if (surr_m_depe_stat_val_list_m[0] > 1):
                        raise Exception('Impossibly high value of the KS distance')

                for i_free_stat in range(num_free_stats):
                    #Calculate values of discriminating statistic (model-free statistics) for observed sequence and surrogate sequences:
                    obs_stat = free_stat_val_list[i_free_stat]
                    abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                    if (abs_obs_stat == np.inf):
                        abs_obs_stat = 0
                    obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                    if np.isnan(obs_stat):
                        print(f'There may be a problem with model-free statistic no. {i_free_stat} (observed)')
                    for i_method in range(num_methods):
                        if (i_method == 0):#Constrained surrogates
                            stat_val_surr_m_list = c_surr_m_free_list_list[:, i_free_stat]
                        elif (i_method == 1):#Typical surrogates
                            stat_val_surr_m_list = t_surr_m_free_list_list[:, i_free_stat]
                        stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                        if np.any(np.isnan(stat_val_surr_m_list)):
                            print(f'There may be a problem with model-free statistic no. {i_free_stat} (surrogate)')
                        #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                        rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                        rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                        r = random.randint(rankMin, rankMax)
                        q = (r - 0.5)/(num_surr + 1)
                        free_quantile_power_list_list[i_method, i_model, i_test, i_free_stat] = q
                        free_quantile_list_list_list[i_method, i_process, i_model, i_test, i_free_stat] = q
                        if (np.isnan(q)):
                            raise Exception('Calculated quantile q is not a number.')
                            
                for i_depe_stat in range(num_depe_stats):
                    #Calculate values of discriminating statistic (model-dependent statistics) for observed sequence and surrogate sequences:
                    obs_stat = depe_stat_val_list[i_depe_stat]
                    abs_obs_stat = abs(obs_stat)#Add small random perturbations to avoid ties
                    if (abs_obs_stat == np.inf):
                        abs_obs_stat = 0
                    obs_stat = obs_stat + 10**-6*abs_obs_stat*(np.random.uniform() - 0.5)
                    if np.isnan(obs_stat):
                        print(f'There may be a problem with model-dependent statistic no. {i_depe_stat} (observed)')
                    for i_method in range(num_methods):
                        if (i_method == 0):#Constrained surrogates
                            stat_val_surr_m_list = c_surr_m_depe_list_list[:, i_depe_stat]
                        elif (i_method == 1):#Typical surrogates
                            stat_val_surr_m_list = t_surr_m_depe_list_list[:, i_depe_stat]
                        stat_val_surr_m_list = stat_val_surr_m_list + 10**-6*abs_obs_stat*(np.random.uniform(size=num_surr) - 0.5)#Add small random perturbations to avoid ties
                        if np.any(np.isnan(stat_val_surr_m_list)):
                            print(f'There may be a problem with model-dependent statistic no. {i_depe_stat}  (surrogate)')
                        #Work out rank of observed statistic (position of statistic corresponding to observed sequence when statistics corresponding to observed sequence and surrogate sequences are ranked from smallest to largest)
                        rankMin = 1 + sum(stat_val_surr_m_list < obs_stat)
                        rankMax = 1 + sum(stat_val_surr_m_list <= obs_stat)
                        r = random.randint(rankMin, rankMax)
                        q = (r - 0.5)/(num_surr + 1)
                        depe_quantile_power_list_list[i_method, i_model, i_test, i_depe_stat] = q
                        depe_quantile_list_list_list[i_method, i_process, i_model, i_test, i_depe_stat] = q
                        if (np.isnan(q)):
                            raise Exception('Calculated quantile q is not a number.')

        end_time_process = timer()
        total_time_process = end_time_process - start_time_process
        process_time_list = process_time_list + [total_time_process]
        print('That took ' + str(total_time_process) + 'sec.') # Time in seconds, e.g. 5.38091952400282
        
    save_str = save_str_0 + process_str

    free_stat_name_list = [stat.__name__ for stat in free_stat_list]
    depe_stat_name_list = [stat.__name__ for stat in depe_stat_list]

    save_dict = {'free_quantile_list_list_list':free_quantile_list_list_list.tolist(),
                 'depe_quantile_list_list_list':depe_quantile_list_list_list.tolist(),
                 'surr_time_list_list_list':surr_time_list_list_list.tolist(),
                 'param_fit_time_list_list_list':param_fit_time_list_list_list.tolist(),
                 'test_type':test_type,
                 'lower_cutoff':lower_cutoff,
                 'N':N,
                 'num_trans':num_trans,
                 'num_surr':num_surr,
                 'num_tests':num_tests,
                 'num_methods':num_methods,
                 'process_list':process_list,
                 'process_dict':process_dict,
                 'process_str':process_str,
                 'model_list':model_list,
                 'save_str':save_str,
                 'process_time_list':process_time_list,
                 'free_stat_name_list':free_stat_name_list,
                 'depe_stat_name_list':depe_stat_name_list,
    }

    save('./results/hyp-test/benchmarking/' + save_str, save_dict)

N = 1024, lower cut-off = 1
1000 tests, each with 19 constrained and typical surrogates; constrained surrogates use 20480 transitions
Process is powerlaw
True [alpha] = [1.5]
That took 86452.7295177sec.
Process is lognorm
True [mu/alpha, sigma] = [-1, 1]
That took 80688.76535280001sec.
Process is expon
True [lambda] = [1]
That took 82313.16184700001sec.
Process is truncnorm
True [mu/lambda, sigma] = [-1, 1]
That took 82806.79136929999sec.
Process is uniform
True [b] = [9]
That took 77941.48520489997sec.
